### PROJETO AUTOMAÇÃO DO FATURAMENTO CÍVEL ###

### IMPORTANTE SE ATENTAR A MUDANÇA DE DATAS NAS CÉLULAS 28 E 31

In [0]:
%run "/Workspace/Jurídico/Normalizador_de_dados/Normalizador"

In [0]:
dbutils.widgets.text("Mes", "", "Data Bases Ativo e Encerrrado")
Mes = dbutils.widgets.get("Mes")

In [0]:
base_Ativos = f'/Volumes/databox/juridico_comum/arquivos/financeiro/Faturamento_Civel/Bases_Mensais_Ativo_Encerrado/CIVEL_GERENCIAL_(ATIVOS)-{Mes}.xlsx'
base_encerrados = f'/Volumes/databox/juridico_comum/arquivos/financeiro/Faturamento_Civel/Bases_Mensais_Ativo_Encerrado/CIVEL_GERENCIAL_(ENCERRADOS)-{Mes}.xlsx'
base_gerencial_final = f'/Volumes/databox/juridico_comum/arquivos/financeiro/Faturamento_Civel/Base_Final_Mensal/'
base_modelo_gerencial = f'/Volumes/databox/juridico_comum/arquivos/financeiro/Faturamento_Civel/Base_Gerencial_Modelo/CIVEL_GERENCIAL.xlsx'

In [0]:
# carrega as bases para o codigo

df_ativos = pd.read_excel(
    base_Ativos,
    sheet_name='CÍVEL',
    header=5 
)

df_encerrados = pd.read_excel(
    base_encerrados,
    sheet_name='CÍVEL',
    header=5
)

In [0]:
print("O df_ativos possui ativos:", df_ativos.shape)
print("O df_encerrados possui encerrados:", df_encerrados.shape)

##### 2º Tratamento das bases, realizando os filtros necessarios #####

In [0]:
#Converte a coluna DATA DE REGISTRO DO ENCERRAMENTO para datime sem as horas e ja renomeia para DATA ENCERRAMENTO no df_
 
df_encerrados['DATA CONVERTIDA'] = pd.to_datetime(df_encerrados['DATA DE REGISTRO DO ENCERRAMENTO'])

df_encerrados['DATA ENCERRAMENTO'] = df_encerrados['DATA CONVERTIDA'].dt.date

In [0]:
df_encerrados['DATA CONVERTIDA_1'] = pd.to_datetime(df_encerrados['DATA REGISTRADO'])

df_encerrados['DATA REGISTRO'] = df_encerrados['DATA CONVERTIDA_1'].dt.date

In [0]:
#Converte a coluna DATA DE REGISTRO DO ENCERRAMENTO para datime sem as horas e ja renomeia para DATA ENCERRAMENTO no df_ativo
 
df_ativos['DATA CONVERTIDA'] = pd.to_datetime(df_ativos['DATA DE REGISTRO DO ENCERRAMENTO'])

df_ativos['DATA ENCERRAMENTO'] = df_ativos['DATA CONVERTIDA'].dt.date

In [0]:
df_ativos['DATA CONVERTIDA_1'] = pd.to_datetime(df_ativos['DATA REGISTRADO'])

df_ativos['DATA REGISTRO'] = df_ativos['DATA CONVERTIDA_1'].dt.date

In [0]:
# APAGA AS COLUNAS ANTIGAS PARA NÃO IMPACTAR NA QUANTIDADE FINAL DE COLUNAS

df_encerrados.drop(columns=['DATA CONVERTIDA', 'DATA DE REGISTRO DO ENCERRAMENTO','DATA CONVERTIDA_1','DATA REGISTRADO'], inplace=True)
df_ativos.drop(columns=['DATA CONVERTIDA', 'DATA DE REGISTRO DO ENCERRAMENTO','DATA CONVERTIDA_1','DATA REGISTRADO'], inplace=True)


In [0]:
# Ordena no df_encerrado

Ordena_colunas = df_encerrados.columns.tolist()
Ordena_colunas.remove('DATA REGISTRO')
Ordena_colunas.insert(8,'DATA REGISTRO')
df_encerrados = df_encerrados[Ordena_colunas]

In [0]:
## Ordena no df_encerrado

Ordena_colunas = df_encerrados.columns.tolist()
Ordena_colunas.remove('DATA ENCERRAMENTO')
Ordena_colunas.insert(56,'DATA ENCERRAMENTO')
df_encerrados = df_encerrados[Ordena_colunas]

In [0]:
# Ordena no df_ativo

Ordena_colunas = df_ativos.columns.tolist()
Ordena_colunas.remove('DATA REGISTRO')
Ordena_colunas.insert(8,'DATA REGISTRO')
df_ativos = df_ativos[Ordena_colunas]

In [0]:
## Ordena no df_ativo

Ordena_colunas = df_ativos.columns.tolist()
Ordena_colunas.remove('DATA ENCERRAMENTO')
Ordena_colunas.insert(56,'DATA ENCERRAMENTO')
df_ativos = df_ativos[Ordena_colunas]

In [0]:
df_encerrados.shape

#### 3 ºAPLICA AS REGRAS DE FORMATAÇÃO ####

In [0]:
#FILTRA POR CIVIL MASSA E APAGA O CIVEL ESTRATEGICO

df_ativos = df_ativos[df_ativos['ÁREA DO DIREITO'] != 'CÍVEL ESTRATÉGICO']
df_encerrados = df_encerrados[df_encerrados['ÁREA DO DIREITO'] != 'CÍVEL ESTRATÉGICO']

In [0]:
print("O df possui ativos:", df_ativos.shape)
print("O df possui encerrados:", df_encerrados.shape)

In [0]:
# Define a data limite <==== ALTERE NO PROXIMO FATURAMENTO da coluna DATA REGISTRADO
data_limite_str = '2026-02-16'

# 2. Convert the string to a Pandas Timestamp
data_limite = pd.to_datetime(data_limite_str)

# 3. Ensure the column is also in datetime format (Safety step)
# This fixes issues where the column might be mixed types or Python 'date' objects
df_ativos['DATA REGISTRO'] = pd.to_datetime(df_ativos['DATA REGISTRO'])

# Cria um novo DataFrame (df_filtrado) apenas com as linhas
# onde 'DATA REGISTRO' é menor ou igual à data limite.
df_filtrado = df_ativos[df_ativos['DATA REGISTRO'] <= data_limite].copy()

df_ativos = df_filtrado

In [0]:
df_ativos = df_ativos[(df_ativos['ESCRITÓRIO EXTERNO'] != 'ARYSTOBULO FREITAS ADVOGADOS') & (df_ativos['ESCRITÓRIO EXTERNO'] != 'ESCRITÓRIO INTERNO')]

In [0]:
escritorios_excluir = ['MASCARENHAS BARBOSA ADVOGADOS (CÍVEL/REG)',
                       'ALMEIDA SANTOS SOCIEDADE DE ADVOGADOS',
                       'FRAGATA E ANTUNES ADVOGADOS',
                       'ERNESTO BORGES ADVOGADOS',
                       'SIQUEIRA CASTRO ADVOGADOS (CÍVEL)',
                       'OLIVEIRA E ANTUNES ADVOGADOS ASSOCIADOS SC',
                       'A DEFINIR',
                       'ESCRITÓRIO INTERNO',
                       'FREIRE, GERBASI, BITTENCOURT E MACEDO',
                       'VISEU ADVOGADOS',
                       'LEE, BROCK, CAMARGOS ADVOGADOS (CÍVEL)',
                       'ESCRITÓRIO EFICIÊNCIA JURÍDICA']

In [0]:
condicao_excluir = (df_encerrados['ÁREA DO DIREITO'] == 'CÍVEL MASSA') & \
                    (df_encerrados['ESCRITÓRIO EXTERNO'].isin(escritorios_excluir))

In [0]:
df_encerrados = df_encerrados[~condicao_excluir]

In [0]:
df_encerrados.shape

In [0]:
#data_inicio_vigencia = pd.to_datetime("2025-12-16").date()   # <==== ALTERE NO PROXIMO FATURAMENTO
#data_final_vigencia = pd.to_datetime('2026-01-15').date()      # <==== ALTERE NO PROXIMO FATURAMENTO

#filtro_intervalo = (df_encerrados['DATA ENCERRAMENTO']  >= data_inicio_vigencia) & (df_encerrados['DATA ENCERRAMENTO'] <= data_final_vigencia)

#df_encerrados = df_encerrados[filtro_intervalo]

In [0]:
# Garante que a coluna esteja em datetime
df_encerrados['DATA ENCERRAMENTO'] = pd.to_datetime(df_encerrados['DATA ENCERRAMENTO'])

# Definindo o intervalo que você QUER MANTER (16/Dez a 15/Jan)
data_inicio = '2026-01-16'
data_fim = '2026-02-16'

# Filtra mantendo apenas o que está DENTRO desse intervalo
# Note que removemos o '~' (til), pois agora estamos selecionando o positivo
df_encerrados = df_encerrados[df_encerrados['DATA ENCERRAMENTO'].between(data_inicio, data_fim)]

In [0]:
print("O df possui ativos:", df_ativos.shape)
print("O df possui encerrados:", df_encerrados.shape)

In [0]:
# JUNTA AS DUAS BASES TRATADAS E GERA A BASE GERENCIAL

df_gerencial = pd.concat([df_ativos, df_encerrados], ignore_index=True)

In [0]:
print("df_gerencial gerado com:", df_gerencial.shape)

In [0]:
df_gerencial.rename(columns={'DATA ENCERRAMENTO': 'DATA DE REGISTRO DO ENCERRAMENTO'}, inplace=True)

In [0]:
df_gerencial.dtypes

#### 4º APLICA A BASE NO MODELO DO ARQUIVO GERENCIAL E SALVA NA PASTA DA REDE ####

In [0]:
df_modelo = pd.read_excel(
    base_modelo_gerencial,
    sheet_name='CÍVEL',
    header=5
)

In [0]:
# Cria o nome final do arquivo usando uma f-string
nome_arquivo_dinamico = f"CIVEL_GERENCIAL - {Mes}.xlsx"

# Junta a pasta de destino com o nome dinâmico do arquivo
caminho_saida_dinamico = os.path.join(base_gerencial_final, nome_arquivo_dinamico)

In [0]:
# Célula 2: Importar bibliotecas e executar o processo
import pandas as pd
import openpyxl
from openpyxl.utils.dataframe import dataframe_to_rows
import shutil
import os

# Caminho para o seu arquivo de modelo original
caminho_modelo_original = "/Volumes/databox/juridico_comum/arquivos/financeiro/Faturamento_Civel/Base_Gerencial_Modelo/CIVEL_GERENCIAL_NAO_MARCADO.xlsx"

# Caminho final onde o novo arquivo preenchido será salvo
nome_arquivo_final = f"CIVEL_GERENCIAL - {Mes}.xlsx"
caminho_final = f"/Volumes/databox/juridico_comum/arquivos/financeiro/Faturamento_Civel/Base_Final_Mensal/{nome_arquivo_final}"

# Configurações para a colagem dos dados
NOME_DA_ABA = "CÍVEL" 
LINHA_INICIAL = 7  
COLUNA_INICIAL = 1  

caminho_local_modelo = f"/tmp/modelo_temp.xlsx"
caminho_local_final = f"/tmp/{nome_arquivo_final}"

try:
    print(f"Copiando modelo de '{caminho_modelo_original}' para '{caminho_local_modelo}'...")
    shutil.copyfile(caminho_modelo_original, caminho_local_modelo)

    # --- Passo 2: O DataFrame já está no formato Pandas ---
    print("O DataFrame já está no formato Pandas. Atribuindo à variável de trabalho...")
    pd_gerencial = df_gerencial 

    print(f"Abrindo o modelo Excel em '{caminho_local_modelo}'...")
    workbook = openpyxl.load_workbook(caminho_local_modelo)
    
    if NOME_DA_ABA in workbook.sheetnames:
        worksheet = workbook[NOME_DA_ABA]
        print(f"Aba '{NOME_DA_ABA}' selecionada.")
    else:
        raise ValueError(f"A aba '{NOME_DA_ABA}' não foi encontrada no arquivo modelo!")

    print(f"Escrevendo dados na aba a partir da célula ({LINHA_INICIAL}, {COLUNA_INICIAL})...")
    rows = dataframe_to_rows(pd_gerencial, index=False, header=False)
    
    for r_idx, row in enumerate(rows, LINHA_INICIAL):
        for c_idx, value in enumerate(row, COLUNA_INICIAL):
            worksheet.cell(row=r_idx, column=c_idx, value=value)

    print(f"Salvando o arquivo preenchido em '{caminho_local_final}'...")
    workbook.save(caminho_local_final)
    workbook.close()

    print(f"Movendo arquivo final para '{caminho_final}'...")
    shutil.move(caminho_local_final, caminho_final)
    
    print("🎉 Processo concluído com sucesso!")

except Exception as e:
    print(f"❌ Ocorreu um erro: {e}")

finally:
    if os.path.exists(caminho_local_modelo):
        os.remove(caminho_local_modelo)
    if os.path.exists(caminho_local_final):
        os.remove(caminho_local_final)
    print("Limpeza dos arquivos temporários concluída.")